# Synthetic Data Generator Pipeline
### Step 3 : Synthetic Observations Timestamping Stage

In [ ]:
from utils.postgres_util import (
    get_engine_from_env, 
    read_sql_dataframe,
)

from utils.synthetic.pipeline.timestamp_stage_writer import (
    ensure_simulation_timing_config_table,
    insert_simulation_timing_config,
    build_observations_timestamped_stage,
    validate_observations_timestamped_stage,
)


In [ ]:
SCHEMA = str("capstone")
DATASET_ID = str("pump_synthetic_v1")
RUN_ID = str("premelt_run_001")

IF_EXISTS_FLAG = str("replace")
CHUNK_SIZE = int(250000)

SIMULATION_TIME_CONFIG_TABLE_NAME = str("simulation_timing_config")
SIMULATION_START_DATETIME = str("2026-04-05 08:00:00+00:00")
SIMULATION_SAMPLING_INTERVAL_SECONDS = float(60.0)

TIMESTAMPED_SOURCE_TABLE_NAME = str("synthetic_observations_premelt_stage")
TIMESTAMPED_DESTINATION_TABLE_NAME = str("synthetic_observations_timestamped_stage")


---

In [3]:
engine = get_engine_from_env()


---

In [4]:

ensure_simulation_timing_config_table(
    engine=engine,
    schema=SCHEMA,
    table_name=SIMULATION_TIME_CONFIG_TABLE_NAME,
)


'simulation_timing_config'

---

In [5]:

insert_simulation_timing_config(
    engine=engine,
    dataset_id=DATASET_ID,
    run_id=RUN_ID,
    simulation_start_datetime=SIMULATION_START_DATETIME,
    sampling_interval_seconds=SIMULATION_SAMPLING_INTERVAL_SECONDS,
    schema=SCHEMA,
    table_name=SIMULATION_TIME_CONFIG_TABLE_NAME,
    set_active=True,
    deactivate_existing_for_run=True,
)

print("Timing config ready.")

Timing config ready.


---

In [6]:
timestamped_table_name = build_observations_timestamped_stage(
    engine=engine,
    schema=SCHEMA,
    source_table=TIMESTAMPED_SOURCE_TABLE_NAME,
    target_table=TIMESTAMPED_DESTINATION_TABLE_NAME,
    timing_config_table=SIMULATION_TIME_CONFIG_TABLE_NAME,
    dataset_id=DATASET_ID,
    run_id=RUN_ID,
    if_exists=IF_EXISTS_FLAG,
    chunk_size=CHUNK_SIZE,
)

print("Built table:", timestamped_table_name)


[chunk] 1 | source rows 0 to 71,999
[timestamp] wrote chunk 1 with 72,000 rows
Built table: synthetic_observations_timestamped_stage


---

In [ ]:
validation_dataframe = validate_observations_timestamped_stage(
    engine=engine,
    schema=SCHEMA,
    table_name=TIMESTAMPED_DESTINATION_TABLE_NAME,
)

display(validation_dataframe)

---

In [9]:

sample_dataframe = read_sql_dataframe(
    engine,
    """
    SELECT
        observation_index,
        observation_timestamp,
        generated_row_id,
        stream_state,
        phase,
        meta_episode_id,
        sensor_00,
        sensor_01
    FROM capstone.synthetic_observations_timestamped_stage
    ORDER BY observation_index
    LIMIT 10
    """
)

display(sample_dataframe)


,observation_index,observation_timestamp,generated_row_id,stream_state,phase,meta_episode_id,sensor_00,sensor_01
0,1,2026-03-19 08:00:00+00:00,premelt_run_001_obs_000000000001,normal,normal,0,2.515297,48.692947
1,2,2026-03-19 08:01:00+00:00,premelt_run_001_obs_000000000002,normal,normal,0,2.302084,44.102592
2,3,2026-03-19 08:02:00+00:00,premelt_run_001_obs_000000000003,normal,normal,0,2.492543,45.462963
3,4,2026-03-19 08:03:00+00:00,premelt_run_001_obs_000000000004,normal,normal,0,2.519502,49.937184
4,5,2026-03-19 08:04:00+00:00,premelt_run_001_obs_000000000005,normal,normal,0,2.198398,49.545707
5,6,2026-03-19 08:05:00+00:00,premelt_run_001_obs_000000000006,normal,normal,0,1.650174,48.909007
6,7,2026-03-19 08:06:00+00:00,premelt_run_001_obs_000000000007,normal,normal,0,NaN,50.450918
7,8,2026-03-19 08:07:00+00:00,premelt_run_001_obs_000000000008,normal,normal,0,2.214998,49.661356
8,9,2026-03-19 08:08:00+00:00,premelt_run_001_obs_000000000009,normal,normal,0,2.378135,50.834364
9,10,2026-03-19 08:09:00+00:00,premelt_run_001_obs_000000000010,normal,normal,0,2.291894,50.491806


---


In [10]:
timestamp_check_dataframe = read_sql_dataframe(
    engine,
    """
    SELECT
        observation_index,
        observation_timestamp,
        stream_state,
        phase,
        meta_episode_id
    FROM capstone.synthetic_observations_timestamped_stage
    ORDER BY observation_index
    LIMIT 10
    """
)

display(timestamp_check_dataframe)


,observation_index,observation_timestamp,stream_state,phase,meta_episode_id
0,1,2026-03-19 08:00:00+00:00,normal,normal,0
1,2,2026-03-19 08:01:00+00:00,normal,normal,0
2,3,2026-03-19 08:02:00+00:00,normal,normal,0
3,4,2026-03-19 08:03:00+00:00,normal,normal,0
4,5,2026-03-19 08:04:00+00:00,normal,normal,0
5,6,2026-03-19 08:05:00+00:00,normal,normal,0
6,7,2026-03-19 08:06:00+00:00,normal,normal,0
7,8,2026-03-19 08:07:00+00:00,normal,normal,0
8,9,2026-03-19 08:08:00+00:00,normal,normal,0
9,10,2026-03-19 08:09:00+00:00,normal,normal,0


---

In [11]:
# Dispose Engine
engine.dispose()